# pic0rick RP2350 DSP firmware — step-by-step example

Notebook version of `example_dsp.py`. Targets the **`-DDSP`** firmware build
(see `docs/dsp_test_guide.md`). Run the cells top to bottom.

It shows the **raw text replies from the Pico** (parameters + per-stage DSP
microsecond times) alongside the decoded captures. Uses `Pic0rick.status()`,
`.read_raw()` (8000 raw ADC), `.read_fft()` (4096-sample Hilbert envelope), and
the pulser.

Requires: `pyserial`, `numpy`, `matplotlib`.

## 1. Imports

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from pic0rick.device import Pic0rick
from pic0rick import dsp

## 2. Connect

Leave `PORT = None` to auto-detect the USB-CDC port, or set it explicitly
(e.g. `"/dev/ttyACM0"`, `"COM7"`).

In [ ]:
PORT = None
probe = Pic0rick(port=PORT, verbose=False)

## 3. Version

The Pico's `version` reply, verbatim.

In [ ]:
print(probe.version()['raw'])

## 4. Status

First the Pico's exact `status` line, then a readable breakdown (parameters +
per-stage DSP times). `status()` only works on a DSP build (else `ValueError` —
flash a `rp2350-*-dsp` UF2, see the test guide).

In [ ]:
st = probe.status()
print('Pico reply:')
print(' ', st['raw'])
print()
print(dsp.describe_status(st))

## 5. Set the TGC gain

The DSP build's command is `dac write <n>` (0..1023); its reply is printed.
(The stdio build uses `write dac`, i.e. `Pic0rick.dac()`.)

In [ ]:
probe.ser.write(b'dac write 300\n')
print('Pico reply:', ''.join(b.decode('utf-8', 'replace') for b in probe.sread()).strip())

## 6. `read_raw` — 8000 raw ADC samples (no FFT)

`.samples()` returns a numpy `uint16` array; the CRC is already checked.
`probe.last_reply` is the Pico's `OK capture started ...` text.

In [ ]:
frame = probe.read_raw()
raw = frame.samples()
print('Pico reply:', probe.last_reply)
print('header    : samples=%d bytes=%d rate=%d Hz dc_mean=%.1f seq=%d' % (
    frame.header.sample_count, frame.header.payload_bytes,
    frame.header.sample_rate_hz, frame.header.adc_dc_mean, frame.header.sequence))
print('data      : min/mean/max = %d/%d/%d' % (int(raw.min()), int(raw.mean()), int(raw.max())))
plt.figure(figsize=(9, 3))
plt.plot(raw)
plt.title('read_raw (8000 ADC samples)')
plt.xlabel('sample'); plt.ylabel('ADC code (10-bit)')
plt.show()

## 7. `read_fft` — 4096-sample Hilbert envelope

After the capture, re-read `status` to show the **per-stage DSP times** for the
frame that just ran.

In [ ]:
frame = probe.read_fft()
env = frame.samples()
print('Pico reply:', probe.last_reply)
print('header    : samples=%d bytes=%d envelope_peak=%.1f dc_mean=%.1f' % (
    frame.header.sample_count, frame.header.payload_bytes,
    frame.header.envelope_peak, frame.header.adc_dc_mean))
print()
print(dsp.describe_status(probe.status()))   # DSP stage/us times for this frame
plt.figure(figsize=(9, 3))
plt.plot(env)
plt.title('read_fft (4096-pt Hilbert envelope)')
plt.xlabel('sample'); plt.ylabel('envelope (ADC counts)')
plt.show()

## 8. Pulsed acquisition

Configure and arm the pulser (replies shown), capture an envelope, and compare
with the idle one.

In [ ]:
def send(cmd):
    probe.ser.write((cmd + '\n').encode('ascii'))
    reply = ''.join(b.decode('utf-8', 'replace') for b in probe.sread()).strip()
    print(cmd, '->', reply)
    return reply

send('pulse config 96 6000 96 neg-first')
send('pulser arm')
env_pulsed = probe.read_fft().samples()
print('Pico reply:', probe.last_reply)
send('pulser disarm')
print('idle peak %.1f  |  pulsed peak %.1f' % (float(env.max()), float(env_pulsed.max())))
plt.figure(figsize=(9, 3))
plt.plot(env, label='idle')
plt.plot(env_pulsed, label='pulser armed')
plt.title('read_fft envelope: idle vs pulser armed')
plt.xlabel('sample'); plt.ylabel('envelope'); plt.legend()
plt.show()

## 9. (Optional) save the captures

In [ ]:
np.save('raw.npy', raw)
np.save('envelope.npy', env)
print('saved raw.npy and envelope.npy')

---
See `docs/dsp_test_guide.md` for the full command set, `pic0rick.dsp` for the
frame/protocol details, and `python/example_dsp.py` for the same flow as a
runnable script.